<a href="https://colab.research.google.com/github/Microblast56/Flyrank_ml_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Microblast56/Flyrank_ml_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## The rule, in plain words

Review pages with meaningful historical search traffic, weak average search position, and low click-through rate (CTR). A page receives a positive score only if it meets all three conditions. Among selected pages, higher historical impressions determine review priority.

The rule uses February 2026 information to rank pages for review. March 2026 is reserved for retrospective evaluation. The score is a priority heuristic, not a prediction of refresh success.

### Conditions

- `meaningful_traffic`: At least 600 February impressions.
- `weak_position`: Average February search position is 20 or worse.
- `low_ctr`: February CTR is at most 1%.

### Reason codes

The three reason codes above can be combined, for example:
`meaningful_traffic+weak_position+low_ctr`.

`no_priority_signal` means none of the three conditions was met.

A staleness rule was tested but excluded because the observed update-date distribution left no candidates meeting the original combined conditions. These thresholds are simple starting rules, not optimized model parameters.

In [ ]:
%pip -q install duckdb pandas

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from pathlib import Path

# Read the token from Colab Secrets; never paste it into the notebook.
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "Add your Hugging Face Read token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# February = historical features; March = future outcome label.
sql = f"""
WITH daily AS (
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position,
           gsc_data_available
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/month=2026-02/*.parquet',
        hive_partitioning=true
    )

    UNION ALL

    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position,
           gsc_data_available
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/month=2026-03/*.parquet',
        hive_partitioning=true
    )
),
aggregated AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-02-01'
                                 AND DATE '2026-02-28'
             AND gsc_data_available IS TRUE
            THEN COALESCE(gsc_impressions, 0) ELSE 0
        END) AS imp_b,

        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-02-01'
                                 AND DATE '2026-02-28'
             AND gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0) ELSE 0
        END) AS clk_b,

        AVG(CASE
            WHEN report_date BETWEEN DATE '2026-02-01'
                                 AND DATE '2026-02-28'
             AND gsc_data_available IS TRUE
             AND gsc_avg_position IS NOT NULL
             AND gsc_avg_position > 0
            THEN gsc_avg_position
        END) AS pos_avg_b,

        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-02-01'
                                 AND DATE '2026-02-28'
             AND gsc_data_available IS TRUE
            THEN 1 ELSE 0
        END) AS gsc_days_b,

        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-03-01'
                                 AND DATE '2026-03-31'
             AND gsc_data_available IS TRUE
            THEN COALESCE(gsc_impressions, 0) ELSE 0
        END) AS imp_f,

        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-03-01'
                                 AND DATE '2026-03-31'
             AND gsc_data_available IS TRUE
            THEN 1 ELSE 0
        END) AS gsc_days_f

    FROM daily
    GROUP BY client_hash_id, content_hash_id
),
content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        TRY_CAST(content_updated_date AS DATE) AS updated_date
    FROM read_parquet('{REL}/dim_content.parquet')
)
SELECT
    a.client_hash_id,
    a.content_hash_id,
    a.imp_b,
    a.clk_b,
    a.pos_avg_b,
    a.gsc_days_b,
    a.imp_f,
    a.gsc_days_f,

    DATE_DIFF(
        'day', c.updated_date, DATE '2026-02-28'
    ) AS days_since_update,

    CASE
        WHEN
            CAST(a.imp_f AS DOUBLE) / NULLIF(a.gsc_days_f, 0)
            < 0.8 * (
                CAST(a.imp_b AS DOUBLE) / NULLIF(a.gsc_days_b, 0)
            )
        THEN 1 ELSE 0
    END AS declined_30d

FROM aggregated a
JOIN content c USING (client_hash_id, content_hash_id)

WHERE a.gsc_days_b >= 20
  AND a.gsc_days_f >= 20
  AND a.imp_b > 0
  AND c.updated_date <= DATE '2026-02-28'
"""

df = con.sql(sql).df()

if df.empty:
    raise ValueError("No eligible rows returned. Check the data access and filters.")

LABEL_COLUMN = "declined_30d"

print("DATA PREPARATION COMPLETE")
print(f"Eligible content items: {len(df):,}")
print(f"Distinct clients: {df['client_hash_id'].nunique():,}")
print(f"Outcome positive rate: {df[LABEL_COLUMN].mean():.1%}")
print(f"Missing historical average position: {df['pos_avg_b'].isna().sum():,}")
print("Feature window: February 2026")
print("Outcome window: March 2026")
print("GSC availability: IS TRUE")
print("June sample used: No")

display(
    df[
        [
            "imp_b",
            "clk_b",
            "pos_avg_b",
            "days_since_update",
            LABEL_COLUMN,
        ]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DATA PREPARATION COMPLETE
Eligible content items: 17,177
Distinct clients: 21
Outcome positive rate: 25.3%
Missing historical average position: 0
Feature window: February 2026
Outcome window: March 2026
GSC availability: IS TRUE
June sample used: No


,imp_b,clk_b,pos_avg_b,days_since_update,declined_30d
0,52.0,0.0,41.750833,3,0
1,145.0,0.0,19.149863,3,0
2,84.0,0.0,23.518019,3,1
3,251.0,0.0,45.471238,3,0
4,497.0,3.0,3.998111,3,0


## 2. Build the ranked queue

### Rule and score

The baseline uses February 2026 data only. A page is flagged for review when all three conditions hold:

- At least 600 historical impressions.
- Average search position of 20 or worse.
- CTR of at most 1%.

The score equals February impressions for qualifying pages and zero otherwise. Qualifying pages are ranked by historical impressions, so pages with more traffic at stake appear first. This is a transparent heuristic, not a fitted model.

### Measured results

- Eligible pages: 17,177.
- Flagged candidates: 1,256.
- March decline proxy base rate: 25.3%.
- Precision@20: 20.0%.
- Precision@50: 14.0%.
- Precision@100: 8.0%.

The baseline performs below the overall decline rate at each reported cutoff. The current rule is therefore a useful reference point, but not a successful ranking rule on this evaluation slice.

The generated queue is saved to `work/outputs/baseline_action_score.csv`. It contains the ranked action, reason code, and historical measurements, without the March outcome label.

In [ ]:
# ML-07 — Section 2: Build and evaluate the baseline

# Calculate CTR from February historical data.
# 0.01 = 1%, because this is a ratio.
df["ctr_b"] = df["clk_b"] / df["imp_b"].clip(lower=1)

# Transparent rule: February signals only.
MIN_IMPRESSIONS = 600
MIN_POSITION = 20
MAX_CTR = 0.01

has_traffic = df["imp_b"] >= MIN_IMPRESSIONS
weak_position = df["pos_avg_b"] >= MIN_POSITION
low_ctr = df["ctr_b"] <= MAX_CTR

candidate = has_traffic & weak_position & low_ctr

# Candidates get positive scores; others get zero.
# Among candidates, rank by historical impressions.
df["baseline_score"] = candidate.astype(int) * df["imp_b"]

def make_reason_code(row):
    reasons = []

    if row["imp_b"] >= MIN_IMPRESSIONS:
        reasons.append("meaningful_traffic")

    if row["pos_avg_b"] >= MIN_POSITION:
        reasons.append("weak_position")

    if row["ctr_b"] <= MAX_CTR:
        reasons.append("low_ctr")

    return "+".join(reasons) if reasons else "no_priority_signal"

df["reason_code"] = df.apply(make_reason_code, axis=1)
df["action"] = np.where(candidate, "review_page", "no_priority_flag")

# Rank all eligible content items.
ranked = df.sort_values(
    ["baseline_score", "imp_b"],
    ascending=[False, False]
).reset_index(drop=True)

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    order = np.argsort(-scores)
    return float(labels[order[:k]].mean())

labels = ranked[LABEL_COLUMN].astype(int).to_numpy()
scores = ranked["baseline_score"].to_numpy()
base_rate = float(labels.mean())

print("ML-07 BASELINE RESULTS")
print(f"Eligible pages: {len(ranked):,}")
print(f"Flagged candidates: {int(candidate.sum()):,}")
print(f"March outcome base rate: {base_rate:.1%}")
print()

for k in [20, 50, 100]:
    print(f"Precision@{k}: {precision_at_k(scores, labels, k):.1%}")

print("\nReason-code distribution:")
print(ranked["reason_code"].value_counts().to_string())

# Write the ranked queue without the future outcome label.
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue_columns = [
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action",
    "imp_b",
    "pos_avg_b",
    "ctr_b",
]

ranked[queue_columns].to_csv(output_path, index=False)

print(f"\nSaved {len(ranked):,} ranked rows to {output_path}")
print("\nTop 10 ranked rows:")
display(ranked[queue_columns].head(10))

ML-07 BASELINE RESULTS
Eligible pages: 17,177
Flagged candidates: 1,256
March outcome base rate: 25.3%

Precision@20: 20.0%
Precision@50: 14.0%
Precision@100: 8.0%

Reason-code distribution:
reason_code
low_ctr                                     6952
meaningful_traffic+low_ctr                  6687
weak_position+low_ctr                       1497
meaningful_traffic+weak_position+low_ctr    1256
no_priority_signal                           427
meaningful_traffic                           297
weak_position                                 54
meaningful_traffic+weak_position               7

Saved 17,177 ranked rows to work/outputs/baseline_action_score.csv

Top 10 ranked rows:


,client_hash_id,content_hash_id,baseline_score,reason_code,action,imp_b,pos_avg_b,ctr_b
0,client_23a62021009f63c4,content_532d96d500804552,37771.0,meaningful_traffic+weak_position+low_ctr,review_page,37771.0,45.876972,0.000503
1,client_23a62021009f63c4,content_dd5472aea4c7aa91,29467.0,meaningful_traffic+weak_position+low_ctr,review_page,29467.0,40.763580,0.000848
2,client_23a62021009f63c4,content_ea7474d92d9701c3,23605.0,meaningful_traffic+weak_position+low_ctr,review_page,23605.0,33.352488,0.003262
3,client_23a62021009f63c4,content_db1cf8cf217e6051,20825.0,meaningful_traffic+weak_position+low_ctr,review_page,20825.0,38.009811,0.000336
4,client_23a62021009f63c4,content_3418e48c3ce87f76,20383.0,meaningful_traffic+weak_position+low_ctr,review_page,20383.0,32.019756,0.001177
5,client_23a62021009f63c4,content_5bbf9c248a1dadb3,19832.0,meaningful_traffic+weak_position+low_ctr,review_page,19832.0,30.887100,0.001967
6,client_23a62021009f63c4,content_a12d89af10c7513a,18675.0,meaningful_traffic+weak_position+low_ctr,review_page,18675.0,32.572894,0.001285
7,client_23a62021009f63c4,content_e70886c63f95aa1c,18087.0,meaningful_traffic+weak_position+low_ctr,review_page,18087.0,39.204314,0.000774
8,client_23a62021009f63c4,content_fcd637c7229d1e3c,17823.0,meaningful_traffic+weak_position+low_ctr,review_page,17823.0,32.955538,0.000617
9,client_23a62021009f63c4,content_e16985888e9097a5,17722.0,meaningful_traffic+weak_position+low_ctr,review_page,17722.0,31.315158,0.000677


## 3. Top-20 review

The top 20 pages were inspected using their rank, action, reason code, historical score, and March outcome for retrospective evaluation.

Four of the top 20 pages showed the defined March proxy decline. The other 16 did not, giving a retrospective precision of 20.0%, below the overall 25.3% decline rate.

The rule consistently selected pages with meaningful traffic, weak average position, and low CTR. However, these conditions were not sufficient to identify pages that subsequently declined in March.

The queue also shows client concentration: all 20 displayed top-ranked rows belong to the same pseudonymized client. This is a limitation because the rule does not control how many recommendations come from one client.

March outcomes are shown only to review the ranking after the fact. They are not scoring inputs.

In [ ]:
# ML-07 — Section 3: Inspect the top 20 ranked items

top20 = ranked.head(20).copy()

top20["rank"] = np.arange(1, len(top20) + 1)

top20["confidence_note"] = (
    "All three rule conditions pass; "
    "the score is a priority heuristic, not a probability."
)

top20["what_could_make_this_wrong"] = (
    "CTR may be noisy, the query mix may differ, or seasonality and "
    "other changes may explain the weak position."
)

# The March outcome is shown for retrospective review only.
# It was NOT used to calculate the baseline score.
top20["march_outcome_review_only"] = top20[LABEL_COLUMN].map({
    0: "No proxy decline observed",
    1: "Proxy decline observed",
})

review_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "baseline_score",
    "confidence_note",
    "what_could_make_this_wrong",
    "march_outcome_review_only",
]

print("TOP-20 MANUAL REVIEW")
display(top20[review_columns])

hits = int(top20[LABEL_COLUMN].sum())
print(f"\nTop-20 items with observed March proxy decline: {hits}/20")
print(f"Top-20 retrospective precision: {hits / len(top20):.1%}")
print(f"Overall March proxy decline rate: {base_rate:.1%}")

TOP-20 MANUAL REVIEW


,rank,client_hash_id,content_hash_id,action,reason_code,baseline_score,confidence_note,what_could_make_this_wrong,march_outcome_review_only
0,1,client_23a62021009f63c4,content_532d96d500804552,review_page,meaningful_traffic+weak_position+low_ctr,37771.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",Proxy decline observed
1,2,client_23a62021009f63c4,content_dd5472aea4c7aa91,review_page,meaningful_traffic+weak_position+low_ctr,29467.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",No proxy decline observed
2,3,client_23a62021009f63c4,content_ea7474d92d9701c3,review_page,meaningful_traffic+weak_position+low_ctr,23605.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",No proxy decline observed
3,4,client_23a62021009f63c4,content_db1cf8cf217e6051,review_page,meaningful_traffic+weak_position+low_ctr,20825.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",No proxy decline observed
4,5,client_23a62021009f63c4,content_3418e48c3ce87f76,review_page,meaningful_traffic+weak_position+low_ctr,20383.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",Proxy decline observed
5,6,client_23a62021009f63c4,content_5bbf9c248a1dadb3,review_page,meaningful_traffic+weak_position+low_ctr,19832.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",Proxy decline observed
6,7,client_23a62021009f63c4,content_a12d89af10c7513a,review_page,meaningful_traffic+weak_position+low_ctr,18675.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",No proxy decline observed
7,8,client_23a62021009f63c4,content_e70886c63f95aa1c,review_page,meaningful_traffic+weak_position+low_ctr,18087.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",No proxy decline observed
8,9,client_23a62021009f63c4,content_fcd637c7229d1e3c,review_page,meaningful_traffic+weak_position+low_ctr,17823.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",No proxy decline observed
9,10,client_23a62021009f63c4,content_e16985888e9097a5,review_page,meaningful_traffic+weak_position+low_ctr,17722.0,All three rule conditions pass; the score is a...,"CTR may be noisy, the query mix may differ, or...",No proxy decline observed



Top-20 items with observed March proxy decline: 4/20
Top-20 retrospective precision: 20.0%
Overall March proxy decline rate: 25.3%


### What the results show

The rule selected 1,256 of 17,177 eligible pages. However, its retrospective precision was 20.0% at 20, 14.0% at 50, and 8.0% at 100. These figures are below the overall March proxy-decline rate of 25.3%, so the current rule does not outperform the base rate at these ranking cutoffs.

Only 4 of the top 20 pages showed the March proxy decline. The other 16 are weak picks for this particular outcome: their February signals looked like review opportunities, but their March impressions did not meet the defined decline condition.

The ranking also places the displayed top 20 within a single client. Because the baseline has no client-diversity constraint, large-volume clients can dominate the review queue.

### Leakage audit

- **Historical features:** The score uses February impressions, February average search position, and February CTR only.
- **Future window:** March impressions are used to construct the outcome label and evaluate the ranking, not to compute the score or select candidates.
- **Product flags:** No product decision flags are used in the scoring rule.
- **Identifiers:** Client and content hashes identify records in the exported review queue; they are not scoring inputs.
- **Output:** The CSV contains the ranking, action, reason code, and historical measurements, but excludes the March outcome label.

This is a transparent but weak baseline on the observed March outcome. Keep it as the reference point for later model comparison rather than claiming that it predicts decline successfully. Any revised rule should be evaluated separately to avoid tuning to the same outcome window.

In [ ]:
# ML-07 — Section 4: Weak picks and leakage audit

# Recreate the top 20 for a reproducible review.
top20_audit = ranked.head(20).copy()
top20_audit["rank"] = np.arange(1, len(top20_audit) + 1)

weak_picks = top20_audit[
    top20_audit[LABEL_COLUMN] == 0
].copy()

print("WEAK-PICK REVIEW")
print(f"Top-20 rows reviewed: {len(top20_audit)}")
print(
    "Top-20 with March proxy decline:",
    f"{int(top20_audit[LABEL_COLUMN].sum())}/{len(top20_audit)}"
)
print(f"Weak picks (no March proxy decline): {len(weak_picks)}")
print(
    "Overall March proxy-decline rate:",
    f"{base_rate:.1%}"
)

print("\nWeak picks — historical ranking signals and observed outcome:")
display(
    weak_picks[
        [
            "rank",
            "reason_code",
            "baseline_score",
            "imp_b",
            "pos_avg_b",
            "ctr_b",
            LABEL_COLUMN,
        ]
    ]
)

# Verify that the score and candidate flag use historical features only.
expected_candidate = (
    (df["imp_b"] >= MIN_IMPRESSIONS)
    & (df["pos_avg_b"] >= MIN_POSITION)
    & (df["ctr_b"] <= MAX_CTR)
)

expected_score = expected_candidate.astype(int) * df["imp_b"]

assert np.array_equal(
    candidate.to_numpy(),
    expected_candidate.to_numpy()
)
assert np.allclose(
    df["baseline_score"].to_numpy(),
    expected_score.to_numpy()
)

# Verify the saved CSV excludes outcome and future-window fields.
saved_columns = pd.read_csv(output_path, nrows=0).columns.tolist()

for forbidden_column in [
    LABEL_COLUMN,
    "imp_f",
    "gsc_days_f",
]:
    assert forbidden_column not in saved_columns, (
        f"Unexpected future/outcome column in saved queue: {forbidden_column}"
    )

print("\nLEAKAGE CHECK")
print("PASS: score uses February impressions, position, and CTR only.")
print("PASS: March outcome is not part of the score calculation.")
print("PASS: saved queue excludes the outcome label and March measurements.")
print("PASS: client/content IDs are not used in the score formula.")
print("PASS: no product decision flags are used in the scoring rule.")

print("\nLimitation: the current baseline ranks below the overall decline rate")
print("at the reported top-20, top-50, and top-100 cutoffs.")
print("Client concentration is not controlled by this rule.")

WEAK-PICK REVIEW
Top-20 rows reviewed: 20
Top-20 with March proxy decline: 4/20
Weak picks (no March proxy decline): 16
Overall March proxy-decline rate: 25.3%

Weak picks — historical ranking signals and observed outcome:


,rank,reason_code,baseline_score,imp_b,pos_avg_b,ctr_b,declined_30d
1,2,meaningful_traffic+weak_position+low_ctr,29467.0,29467.0,40.763580,0.000848,0
2,3,meaningful_traffic+weak_position+low_ctr,23605.0,23605.0,33.352488,0.003262,0
3,4,meaningful_traffic+weak_position+low_ctr,20825.0,20825.0,38.009811,0.000336,0
6,7,meaningful_traffic+weak_position+low_ctr,18675.0,18675.0,32.572894,0.001285,0
7,8,meaningful_traffic+weak_position+low_ctr,18087.0,18087.0,39.204314,0.000774,0
8,9,meaningful_traffic+weak_position+low_ctr,17823.0,17823.0,32.955538,0.000617,0
9,10,meaningful_traffic+weak_position+low_ctr,17722.0,17722.0,31.315158,0.000677,0
10,11,meaningful_traffic+weak_position+low_ctr,17080.0,17080.0,26.356064,0.001932,0
11,12,meaningful_traffic+weak_position+low_ctr,16218.0,16218.0,21.080052,0.000617,0
12,13,meaningful_traffic+weak_position+low_ctr,16088.0,16088.0,26.006530,0.003729,0



LEAKAGE CHECK
PASS: score uses February impressions, position, and CTR only.
PASS: March outcome is not part of the score calculation.
PASS: saved queue excludes the outcome label and March measurements.
PASS: client/content IDs are not used in the score formula.
PASS: no product decision flags are used in the scoring rule.

Limitation: the current baseline ranks below the overall decline rate
at the reported top-20, top-50, and top-100 cutoffs.
Client concentration is not controlled by this rule.


## Self-check

- [x] Every section above is filled — markdown thinking and supporting code
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries in the written outputs
- [x] Claims use careful language: observed, measured, directional, decision-support
- [x] Committed to my repository under `work/notebooks/`